# Índice de Necesidad habitacional
Equipo UBiK2 · Data Jam CVP 2026

**Objetivo:** calcular, a partir de la Encuesta Multipropósito 2021 (SDP), los cuatro indicadores del
Índice de Necesidad para los 95 dominios de la encuesta (`cod_upz_grupo`), la misma llave territorial
del Índice de Presencia CVP (notebook 02).

| Dimensión | Indicador | Pregunta que responde |
|---|---|---|
| Tenencia | `IND_TENENCIA` | ¿Qué parte de los hogares no tiene seguridad jurídica sobre su vivienda? |
| Vivienda | `IND_VIVIENDA` | ¿Qué parte de los hogares está en déficit habitacional? |
| Entorno | `IND_ENTORNO` | ¿Qué parte de los hogares vive en un entorno sin infraestructura de acceso adecuada? |
| Vulnerabilidad | `IND_VULNERABILIDAD` | ¿Qué parte de los hogares tiene jefatura femenina en pobreza monetaria? |

El componente de riesgo no sale de la encuesta: se construye aparte con la capa de amenaza del IDIGER.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q geopandas

import os
import requests
import geopandas as gpd, pandas as pd
print("GeoPandas", gpd.__version__, "| pandas", pd.__version__)

BASE   = "/content/drive/MyDrive/UBiK2_DataJam"
CRUDOS = f"{BASE}/datos_crudos"
PROC   = f"{BASE}/datos_procesados"
GPKG   = f"{PROC}/ubik2.gpkg"
WEB    = f"{PROC}/web"
EM_DIR = f"{CRUDOS}/encuesta_multiproposito_2021"
os.makedirs(EM_DIR, exist_ok=True)
os.makedirs(WEB, exist_ok=True)

In [ ]:
from datetime import date

# Se parte del registro que dejó el notebook 02 para no perder sus fuentes
RUTA_FUENTES = f"{PROC}/fuentes.csv"
fuentes = pd.read_csv(RUTA_FUENTES).to_dict("records") if os.path.exists(RUTA_FUENTES) else []

def registrar_fuente(nombre, entidad, url, archivo, notas=""):
    """Agrega una fuente al registro de procedencia de los datos."""
    fuentes.append({
        "conjunto_de_datos": nombre,
        "entidad": entidad,
        "url": url,
        "archivo": archivo,
        "fecha_descarga": date.today().isoformat(),
        "notas": notas,
    })

## 1. Descarga de la Encuesta Multipropósito 2021

Los cuatro archivos se descargan del conjunto *Encuesta Multipropósito 2021* que la SDP publica en el
portal de Datos Abiertos Bogotá: los microdatos (`em2021.csv`, 1,25 GB, nivel persona), las variables
adicionales (déficit habitacional, pobreza monetaria y otras variables derivadas por la SDP) y sus dos
diccionarios.

El portal corta las conexiones largas, por lo que la descarga se reanuda desde el último byte recibido
hasta completar el tamaño que anuncia el servidor. Si el archivo ya está completo en Drive, no se
vuelve a descargar.

In [ ]:
URL_EM = "https://datosabiertos.bogota.gov.co/dataset/8ac12a95-1415-4812-b343-f07f90608014/resource"
ARCHIVOS_EM = {
    "em2021.csv": f"{URL_EM}/b3fd892e-b9f9-4f34-ac22-6cc50612eac9/download/em2021.csv",
    "20240430_variables_adicionales_2021.csv": f"{URL_EM}/ba44798f-5257-4c5c-8a57-0966da1bb4fa/download/20240430_variables_adicionales_2021.csv",
    "20230220_diccionario_variables_encuesta_em2021.ods": f"{URL_EM}/46ca32a8-0dd3-461e-810a-31781db890f5/download/20230220_diccionario_variables_encuesta_em2021.ods",
    "20240430_diccionario_variables_adicionales_em2021.ods": f"{URL_EM}/5e9a4aa9-75e7-406c-bdee-bc5a7d1831a0/download/20240430_diccionario_variables_adicionales_em2021.ods",
}

def descargar(url, destino, intentos=30):
    """Descarga con reanudación (Range) y verifica el tamaño contra el que anuncia el servidor."""
    total = int(requests.head(url, allow_redirects=True, timeout=60).headers["Content-Length"])
    for _ in range(intentos):
        actual = os.path.getsize(destino) if os.path.exists(destino) else 0
        if actual >= total:
            break
        try:
            with requests.get(url, headers={"Range": f"bytes={actual}-"}, stream=True, timeout=120) as r:
                r.raise_for_status()
                modo = "ab" if r.status_code == 206 else "wb"   # 200 = el servidor ignoró el Range
                with open(destino, modo) as f:
                    for bloque in r.iter_content(8 * 1024 * 1024):
                        f.write(bloque)
        except requests.RequestException as e:
            print(f"  corte de conexión ({type(e).__name__}), se reanuda")
    tam = os.path.getsize(destino)
    assert tam == total, f"{os.path.basename(destino)}: {tam:,} de {total:,} bytes"
    print(f"{os.path.basename(destino)} · {tam / 1e6:,.1f} MB · completo")

for nombre, url in ARCHIVOS_EM.items():
    descargar(url, f"{EM_DIR}/{nombre}")

In [ ]:
registrar_fuente(
    nombre="Encuesta Multipropósito 2021 (microdatos)",
    entidad="Secretaría Distrital de Planeación",
    url=ARCHIVOS_EM["em2021.csv"],
    archivo="datos_crudos/encuesta_multiproposito_2021/em2021.csv",
    notas="Nivel persona; Bogotá y municipios de Cundinamarca; 95 dominios COD_UPZ_GRUPO; "
          "diccionario 20230220_diccionario_variables_encuesta_em2021.ods",
)
registrar_fuente(
    nombre="Variables Adicionales Encuesta Multipropósito 2021",
    entidad="Secretaría Distrital de Planeación",
    url=ARCHIVOS_EM["20240430_variables_adicionales_2021.csv"],
    archivo="datos_crudos/encuesta_multiproposito_2021/20240430_variables_adicionales_2021.csv",
    notas="Déficit habitacional y pobreza monetaria calculados por la SDP; "
          "diccionario 20240430_diccionario_variables_adicionales_em2021.ods",
)

## 2. Lectura de los microdatos

Se leen solo las columnas que usan los indicadores, en bloques de 200.000 filas para no saturar la
memoria de Colab. El archivo viene en codificación Latin-1 y mezcla registros de Bogotá y de
Cundinamarca, así que el filtro se aplica en cada bloque:

- `DPTO = 11`: Bogotá D.C.
- `CLASE = 1`: cabecera urbana (los 95 dominios son urbanos).

| Grupo | Variables |
|---|---|
| Identificación | `DIRECTORIO`, `DIRECTORIO_HOG`, `COD_LOCALIDAD`, `NOMBRE_LOCALIDAD`, `COD_UPZ_GRUPO`, `NOMBRE_UPZ_GRUPO`, `FEX_C` |
| Entorno (capítulo B, vivienda) | `NVCBP1` vía de acceso, `NVCBP2` estado de la vía, `NVCBP3` andén, `NVCBP5` iluminación |
| Tenencia (capítulo C, hogar) | `NHCCP1` forma de tenencia, `NHCCP3` escritura registrada |
| Persona (capítulo E) | `NPCEP5` sexo al nacer, `NPCEP6` parentesco con el jefe del hogar |

In [ ]:
TEXTO = ["NOMBRE_LOCALIDAD", "NOMBRE_UPZ_GRUPO"]
COLUMNAS = ["DIRECTORIO", "DIRECTORIO_HOG", "DPTO", "CLASE", "COD_LOCALIDAD", "COD_UPZ_GRUPO", "FEX_C",
            "NVCBP1", "NVCBP2", "NVCBP3", "NVCBP5", "NHCCP1", "NHCCP3", "NPCEP5", "NPCEP6"] + TEXTO

bloques = []
for b in pd.read_csv(f"{EM_DIR}/em2021.csv", usecols=COLUMNAS, chunksize=200_000,
                     encoding="latin-1", low_memory=False, on_bad_lines="warn"):
    b[["DPTO", "CLASE"]] = b[["DPTO", "CLASE"]].apply(pd.to_numeric, errors="coerce")
    bloques.append(b[(b["DPTO"] == 11) & (b["CLASE"] == 1)])
per = pd.concat(bloques, ignore_index=True)

numericas = [c for c in COLUMNAS if c not in TEXTO]
per[numericas] = per[numericas].apply(pd.to_numeric, errors="coerce")

print(f"Personas en Bogotá urbana: {len(per):,}")
print(f"Hogares: {per['DIRECTORIO_HOG'].nunique():,} | Dominios: {per['COD_UPZ_GRUPO'].nunique()}")
print("Faltantes en llaves:", per[["DIRECTORIO_HOG", "COD_UPZ_GRUPO", "FEX_C"]].isna().sum().to_dict())

## 3. Base a nivel hogar

La encuesta viene a nivel persona, pero los cuatro indicadores son de hogar. Se deja un registro por
`DIRECTORIO_HOG` y se le agregan:

- El sexo del jefe del hogar (`NPCEP6 = 1`).
- Las variables adicionales de la SDP: `N_pobre_monetario` y `N_deficit_habitacional` con sus dos
  componentes. Ese archivo también está a nivel persona, pero sus valores son de hogar.

Antes del cruce se verifica que el factor de expansión y las variables adicionales no cambien dentro
de un mismo hogar; si cambiaran, tomar el primer registro sesgaría el resultado.

In [ ]:
# El factor de expansión debe ser único por hogar
assert per.groupby("DIRECTORIO_HOG")["FEX_C"].nunique().max() == 1

jefes = (per.loc[per["NPCEP6"] == 1, ["DIRECTORIO_HOG", "NPCEP5"]]
            .drop_duplicates("DIRECTORIO_HOG")
            .rename(columns={"NPCEP5": "SEXO_JEFE"}))

ADICIONALES = ["N_pobre_monetario", "N_deficit_habitacional", "N_deficit_cuantitativo", "N_deficit_cualitativo"]
adic = pd.read_csv(f"{EM_DIR}/20240430_variables_adicionales_2021.csv", usecols=["directorio_hog"] + ADICIONALES)
adic = adic.rename(columns={"directorio_hog": "DIRECTORIO_HOG"})
assert (adic.groupby("DIRECTORIO_HOG")[ADICIONALES].nunique(dropna=False) <= 1).all().all()
adic = adic.drop_duplicates("DIRECTORIO_HOG")

hog = (per.drop(columns=["NPCEP5", "NPCEP6"])
          .drop_duplicates("DIRECTORIO_HOG")
          .merge(jefes, on="DIRECTORIO_HOG", how="left", validate="one_to_one")
          .merge(adic, on="DIRECTORIO_HOG", how="left", validate="one_to_one"))

print(f"Hogares: {len(hog):,}")
print(f"Sin jefe identificado: {hog['SEXO_JEFE'].isna().sum():,}")
print(f"Sin cruce con variables adicionales: {hog['N_pobre_monetario'].isna().sum():,}")
print(f"Hogares expandidos (suma FEX_C): {hog['FEX_C'].sum():,.0f}")

## 4. Condiciones de necesidad por hogar

Cada condición se marca con 1 si el hogar la cumple y 0 si no. Además de la condición principal de
cada dimensión se guardan sus componentes, para poder revisar qué explica el valor de cada dominio.

### Tenencia

Mide la inseguridad jurídica sobre la vivienda. Un hogar cumple la condición si:

- vive en *otra forma de tenencia*, que incluye posesión sin título, ocupante de hecho y propiedad
  colectiva (`NHCCP1 = 6`), o
- se declara propietario, con la vivienda pagada o en pago (`NHCCP1` en 1 o 2), pero nadie del hogar
  tiene escritura registrada (`NHCCP3 = 2`).

El usufructo (`NHCCP1 = 5`) no se cuenta: el hogar usa la vivienda con permiso del dueño, que es una
situación distinta a la informalidad en la tenencia.

In [ ]:
hog["C_TEN_SIN_TITULO"] = (hog["NHCCP1"] == 6).astype(int)
hog["C_TEN_PROP_SIN_ESCRITURA"] = (hog["NHCCP1"].isin([1, 2]) & (hog["NHCCP3"] == 2)).astype(int)
hog["C_TENENCIA"] = hog[["C_TEN_SIN_TITULO", "C_TEN_PROP_SIN_ESCRITURA"]].max(axis=1)

### Vivienda

Se usa el déficit habitacional que la SDP calcula con la metodología del DANE
(`N_deficit_habitacional = 1`). Suma dos componentes que se guardan por separado:

- **Cuantitativo** (`N_deficit_cuantitativo`): la vivienda no se puede mejorar y se necesita una nueva
  (tipo de vivienda, paredes, cohabitación, hacinamiento crítico).
- **Cualitativo** (`N_deficit_cualitativo`): la vivienda se puede mejorar (hacinamiento mitigable,
  pisos, cocina, agua, alcantarillado, energía, recolección de basuras).

El cualitativo es el que más se relaciona con las líneas de Mejoramiento de Vivienda de la CVP.

In [ ]:
hog["C_VIVIENDA"] = (hog["N_deficit_habitacional"] == 1).astype(int)
hog["C_VIV_CUANTI"] = (hog["N_deficit_cuantitativo"] == 1).astype(int)
hog["C_VIV_CUALI"] = (hog["N_deficit_cualitativo"] == 1).astype(int)

### Entorno

Mide la falta de infraestructura de acceso a la vivienda, que es lo que interviene Mejoramiento de
Barrios (vías, andenes, escaleras). Un hogar cumple la condición si la edificación donde vive:

- tiene vía de acceso en tierra, sendero o vehicular destapada (`NVCBP1` en 1 o 3), o
- tiene la vía en mal estado (`NVCBP2 = 3`), o
- no tiene andén (`NVCBP3 = 2`).

La iluminación insuficiente o inexistente de la vía (`NVCBP5` en 2 o 3) se guarda como componente,
pero no entra al indicador: afecta a cerca del 16 % de los hogares de la ciudad, también en zonas
consolidadas, y no es una línea de intervención de la CVP.

In [ ]:
hog["C_ENT_VIA_SIN_PAVIMENTO"] = hog["NVCBP1"].isin([1, 3]).astype(int)
hog["C_ENT_VIA_MAL_ESTADO"] = (hog["NVCBP2"] == 3).astype(int)
hog["C_ENT_SIN_ANDEN"] = (hog["NVCBP3"] == 2).astype(int)
hog["C_ENT_ILUMINACION"] = hog["NVCBP5"].isin([2, 3]).astype(int)
hog["C_ENTORNO"] = hog[["C_ENT_VIA_SIN_PAVIMENTO", "C_ENT_VIA_MAL_ESTADO", "C_ENT_SIN_ANDEN"]].max(axis=1)

### Vulnerabilidad

Porcentaje de hogares con jefatura femenina en pobreza monetaria: el jefe del hogar (`NPCEP6 = 1`) es
mujer (`NPCEP5 = 2`) y el hogar es pobre monetario (`N_pobre_monetario = 1`). El denominador son
todos los hogares del dominio.

In [ ]:
hog["C_VULNERABILIDAD"] = ((hog["SEXO_JEFE"] == 2) & (hog["N_pobre_monetario"] == 1)).astype(int)

## 5. Agregación ponderada por dominio

Todos los porcentajes se expanden con `FEX_C`, no con conteos de la muestra:

$$
Indicador_d = \frac{\sum_{h \in d} FEX\_C_h \cdot condición_h}{\sum_{h \in d} FEX\_C_h}
$$

Se guarda también el número de hogares en la muestra de cada dominio, como referencia de la
precisión de cada estimación.

In [ ]:
CONDICIONES = [c for c in hog.columns if c.startswith("C_")]

ponderado = hog[CONDICIONES].mul(hog["FEX_C"], axis=0)
ponderado[["FEX_C", "COD_UPZ_GRUPO"]] = hog[["FEX_C", "COD_UPZ_GRUPO"]]
ponderado["N_HOGARES_MUESTRA"] = 1
suma = ponderado.groupby("COD_UPZ_GRUPO").sum()

indicadores = suma[CONDICIONES].div(suma["FEX_C"], axis=0).round(4)
indicadores.columns = ["IND_" + c[2:] for c in CONDICIONES]
indicadores.insert(0, "HOGARES_EXPANDIDOS", suma["FEX_C"].round(0).astype(int))
indicadores.insert(0, "N_HOGARES_MUESTRA", suma["N_HOGARES_MUESTRA"])

nombres = hog.drop_duplicates("COD_UPZ_GRUPO").set_index("COD_UPZ_GRUPO")[
    ["COD_LOCALIDAD", "NOMBRE_LOCALIDAD", "NOMBRE_UPZ_GRUPO"]]
necesidad = nombres.join(indicadores).reset_index().sort_values("COD_UPZ_GRUPO")
necesidad["COD_UPZ_GRUPO"] = necesidad["COD_UPZ_GRUPO"].astype(int)

PRINCIPALES = ["IND_TENENCIA", "IND_VIVIENDA", "IND_ENTORNO", "IND_VULNERABILIDAD"]
necesidad[["COD_UPZ_GRUPO", "NOMBRE_UPZ_GRUPO"] + PRINCIPALES].head()

### Control de calidad

Se verifica que la tabla tenga una fila por dominio, que la suma de hogares expandidos corresponda a
Bogotá urbana y que cada dominio tenga una muestra suficiente. Se calcula también el total de la
ciudad como referencia.

In [ ]:
assert len(necesidad) == 95 and necesidad["COD_UPZ_GRUPO"].is_unique
assert necesidad[PRINCIPALES].notna().all().all()

print(f"Hogares expandidos: {necesidad['HOGARES_EXPANDIDOS'].sum():,}")
print(f"Muestra por dominio: {necesidad['N_HOGARES_MUESTRA'].min():,} a {necesidad['N_HOGARES_MUESTRA'].max():,} hogares")

bogota = (hog[CONDICIONES].mul(hog["FEX_C"], axis=0).sum() / hog["FEX_C"].sum()).round(4)
bogota.index = ["IND_" + c[2:] for c in CONDICIONES]
print("\nBogotá urbana")
print(bogota.map("{:.1%}".format).to_string())

for c in PRINCIPALES:
    top = necesidad.nlargest(5, c)
    print(f"\n{c}: " + "; ".join(f"{n} {v:.1%}" for n, v in zip(top["NOMBRE_UPZ_GRUPO"], top[c])))

necesidad[PRINCIPALES].describe().round(3)

**Resultado del control de calidad.** La tabla tiene 95 dominios con 84.274 hogares en la muestra
(entre 611 y 1.321 por dominio), que expandidos suman 2,80 millones de hogares de Bogotá urbana.
Todos los hogares cruzan con las variables adicionales.

| Indicador | Bogotá | Mínimo | Mediana | Máximo |
|---|---|---|---|---|
| `IND_TENENCIA` | 9,1 % | 2,3 % | 7,9 % | 25,8 % (Lourdes) |
| `IND_VIVIENDA` | 8,9 % | 0,4 % | 7,4 % | 33,8 % (La Flora) |
| `IND_ENTORNO` | 16,7 % | 2,4 % | 13,3 % | 83,5 % (San Isidro - Patios) |
| `IND_VULNERABILIDAD` | 15,8 % | 3,2 % | 13,1 % | 35,5 % (Lourdes) |

Los dominios con valores altos en varias dimensiones son Lourdes, La Flora, San Isidro - Patios,
Lucero, Alfonso López + Ciudad Usme y Monteblanco + El Tesoro + El Mochuelo.

## 6. Exportación

- `indicadores_necesidad_95_dominios.xlsx`: hoja *Indicadores* (los cuatro indicadores), hoja
  *Detalle* (componentes, muestra y nombres) y hoja *Metodología*.
- Capa `upz_necesidad_95` en `ubik2.gpkg`, con la geometría de `upz_grupos_95` del notebook 02, y su
  GeoJSON liviano para el visor web.

In [ ]:
metodologia = pd.DataFrame({
    "Indicador": PRINCIPALES,
    "Definición": [
        "% hogares con NHCCP1=6 (posesión sin título, ocupante de hecho, propiedad colectiva) "
        "o propietarios (NHCCP1 en 1,2) sin escritura registrada (NHCCP3=2)",
        "% hogares con déficit habitacional DANE (N_deficit_habitacional=1: cuantitativo o cualitativo)",
        "% hogares en edificación con vía sin pavimentar (NVCBP1 en 1,3), vía en mal estado (NVCBP2=3) "
        "o sin andén (NVCBP3=2)",
        "% hogares con jefe (NPCEP6=1) mujer (NPCEP5=2) en pobreza monetaria (N_pobre_monetario=1)",
    ],
})
metodologia["Fuente"] = "Encuesta Multipropósito 2021, SDP. Bogotá urbana (DPTO=11, CLASE=1), hogares ponderados por FEX_C"

XLSX = f"{PROC}/indicadores_necesidad_95_dominios.xlsx"
with pd.ExcelWriter(XLSX) as xw:
    necesidad[["COD_UPZ_GRUPO"] + PRINCIPALES].to_excel(xw, sheet_name="Indicadores", index=False)
    necesidad.to_excel(xw, sheet_name="Detalle", index=False)
    metodologia.to_excel(xw, sheet_name="Metodologia", index=False)
print("Guardado en", XLSX)

In [ ]:
def exportar_web(gdf, nombre, tolerancia_m=3):
    """Simplifica en metros, pasa a WGS84 y guarda un GeoJSON liviano para el front."""
    capa = gdf.copy()
    capa["geometry"] = capa.geometry.simplify(tolerancia_m, preserve_topology=True)
    capa = capa.to_crs(epsg=4326)
    ruta = f"{WEB}/{nombre}.geojson"
    capa.to_file(ruta, driver="GeoJSON", COORDINATE_PRECISION=6)
    print(f"{nombre}.geojson · {len(capa)} elementos · {os.path.getsize(ruta)/1024:.0f} KB")

grupos95 = gpd.read_file(GPKG, layer="upz_grupos_95")
grupos95["cod_upz_grupo"] = grupos95["cod_upz_grupo"].astype(int)

capa_necesidad = grupos95[["cod_upz_grupo", "upz_incluidas", "geometry"]].merge(
    necesidad.rename(columns=str.lower), on="cod_upz_grupo", how="left", validate="one_to_one")
assert len(capa_necesidad) == 95 and capa_necesidad["ind_tenencia"].notna().all()

capa_necesidad.to_file(GPKG, layer="upz_necesidad_95", driver="GPKG")
exportar_web(capa_necesidad, "upz_necesidad_95")
print(gpd.list_layers(GPKG))

In [ ]:
df_fuentes = pd.DataFrame(fuentes).drop_duplicates(subset=["conjunto_de_datos"], keep="last")
df_fuentes.to_csv(RUTA_FUENTES, index=False, encoding="utf-8-sig")
print(f"Guardado en {RUTA_FUENTES} · {len(df_fuentes)} fuentes")
df_fuentes

## Siguiente paso

1. Construir el indicador de riesgo con la capa de amenaza del IDIGER, cruzada por dominio.
2. Normalizar los indicadores de necesidad y de presencia a escala 0-1 y definir sus pesos.
3. Calcular la brecha (`Necesidad − Presencia`) en la tabla maestra de 95 dominios.